# Build team-season xT spatial matrices

This notebook builds a machine-learning-ready dataset where each row is one team in one competition-season, and the features are that team's 16 by 12 xT spatial matrix.

The key conceptual rule is comparability: train one global xT model using all selected matches, then use that shared model to calculate action-level xT values for every team-season. The final clustering matrix is sum-normalized positive xT created by start zone, so it emphasizes spatial style rather than total attacking volume.

## Configuration

Set `DATA_ROOT` to your local StatsBomb open-data folder. The notebook expects `competitions.json`, `matches/{competition_id}/{season_id}.json`, and `events/{match_id}.json`.

In [ ]:
from __future__ import annotations

from collections import defaultdict
from datetime import datetime
import json
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

DATA_ROOT = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GRID_L = 16
GRID_W = 12
PITCH_LENGTH = 120
PITCH_WIDTH = 80
TOTAL_ZONES = GRID_L * GRID_W

USE_OPEN_PLAY_ONLY = True
INCLUDE_CARRIES = True
INCLUDE_PASSES = True
MIN_MATCHES_PER_TEAM_SEASON = 5
MIN_POSITIVE_XT_PER_TEAM_SEASON = 0
SMOOTHING_OPTIONAL = False
SAVE_ACTION_LEVEL_OUTPUT = False

XT_CONVERGENCE_THRESHOLD = 1e-6
XT_MAX_ITERATIONS = 100
RANDOM_SEED = 42

SET_PIECE_PASS_TYPES = {"Corner", "Free Kick", "Goal Kick", "Kick Off", "Throw-in"}

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"Data root: {DATA_ROOT.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")
print(f"Grid: {GRID_L} length bins by {GRID_W} width bins = {TOTAL_ZONES} zones")

## Scan the local StatsBomb dataset

This section reads competition-season metadata and builds a match table. The table is saved to `outputs/match_metadata.csv` and drives both streaming event passes.

In [ ]:
def read_json_file(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def build_match_metadata(data_root: Path) -> pd.DataFrame:
    competitions_path = data_root / "competitions.json"
    if not competitions_path.exists():
        raise FileNotFoundError(f"Could not find {competitions_path}. Set DATA_ROOT to your StatsBomb open-data folder.")

    competitions = read_json_file(competitions_path)
    rows = []
    missing_match_files = []

    for comp in competitions:
        competition_id = comp.get("competition_id")
        season_id = comp.get("season_id")
        matches_path = data_root / "matches" / str(competition_id) / f"{season_id}.json"
        if not matches_path.exists():
            missing_match_files.append(matches_path)
            continue
        try:
            matches = read_json_file(matches_path)
        except Exception as exc:
            warnings.warn(f"Skipping malformed matches file {matches_path}: {exc}")
            continue

        for match in matches:
            home = match.get("home_team", {}) or {}
            away = match.get("away_team", {}) or {}
            rows.append(
                {
                    "competition_id": competition_id,
                    "competition_name": comp.get("competition_name"),
                    "country_name": comp.get("country_name"),
                    "season_id": season_id,
                    "season_name": comp.get("season_name"),
                    "match_id": match.get("match_id"),
                    "match_date": match.get("match_date"),
                    "home_team_id": home.get("home_team_id") or home.get("id"),
                    "home_team_name": home.get("home_team_name") or home.get("name"),
                    "away_team_id": away.get("away_team_id") or away.get("id"),
                    "away_team_name": away.get("away_team_name") or away.get("name"),
                }
            )

    metadata = pd.DataFrame(rows).dropna(subset=["match_id"]).copy()
    if metadata.empty:
        raise ValueError("No matches were found. Check DATA_ROOT and the StatsBomb open-data layout.")
    metadata["match_id"] = metadata["match_id"].astype(int)

    top_comp_seasons = (
        metadata.groupby(["competition_id", "competition_name", "season_id", "season_name"], dropna=False)
        .size()
        .reset_index(name="match_count")
        .sort_values("match_count", ascending=False)
        .head(15)
    )

    print(f"Competitions listed: {len(competitions):,}")
    print(f"Competition-seasons with match files: {metadata[['competition_id', 'season_id']].drop_duplicates().shape[0]:,}")
    print(f"Matches found: {metadata['match_id'].nunique():,}")
    print(f"Unique teams: {pd.concat([metadata['home_team_id'], metadata['away_team_id']]).dropna().nunique():,}")
    print(f"Missing match files: {len(missing_match_files):,}")
    display(top_comp_seasons)

    output_path = OUTPUT_DIR / "match_metadata.csv"
    metadata.to_csv(output_path, index=False)
    print(f"Saved match metadata to {output_path}")
    return metadata


match_metadata = build_match_metadata(DATA_ROOT)

## Event parsing design

Only shots, successful passes, and carries are needed for xT. Shots estimate shot and goal rates; successful passes and carries estimate zone-to-zone transitions. Dribble events are not included because StatsBomb Dribble does not consistently represent a ball movement from one coordinate to another.

In [ ]:
TEAM_SEASON_KEYS = ["competition_id", "competition_name", "season_id", "season_name", "team_id", "team_name"]
TEAM_SEASON_METADATA_COLUMNS = [
    "competition_id", "competition_name", "country_name", "season_id", "season_name", "team_id", "team_name",
    "match_count", "move_action_count", "pass_count", "carry_count", "total_net_xT", "total_positive_xT",
    "total_positive_pass_xT", "total_positive_carry_xT", "is_zero_positive_xT", "below_min_matches",
    "below_min_positive_xT",
]
TOTAL_COLUMNS = [
    "move_action_count", "pass_count", "carry_count", "total_net_xT", "total_positive_xT",
    "total_positive_pass_xT", "total_positive_carry_xT",
]


def get_name(obj, default=None):
    return obj.get("name", default) if isinstance(obj, dict) else default


def valid_xy(location) -> bool:
    if not isinstance(location, list) or len(location) < 2:
        return False
    x, y = location[0], location[1]
    return x is not None and y is not None and np.isfinite(x) and np.isfinite(y)


def point_to_zone(x: float, y: float) -> int | None:
    if x is None or y is None or not (0 <= x <= PITCH_LENGTH and 0 <= y <= PITCH_WIDTH):
        return None
    x_bin = min(int((x / PITCH_LENGTH) * GRID_L), GRID_L - 1)
    y_bin = min(int((y / PITCH_WIDTH) * GRID_W), GRID_W - 1)
    return y_bin * GRID_L + x_bin


def zone_to_xy_bins(zone: int) -> tuple[int, int]:
    return zone % GRID_L, zone // GRID_L


def zone_columns(prefix: str = "z") -> list[str]:
    return [f"{prefix}{zone:03d}" for zone in range(TOTAL_ZONES)]


def event_file_path(match_id: int) -> Path:
    return DATA_ROOT / "events" / f"{int(match_id)}.json"


def iter_match_rows(metadata: pd.DataFrame):
    for row in metadata.itertuples(index=False):
        yield row._asdict()


def empty_processing_counters() -> dict:
    return {
        "start_time": datetime.now(),
        "matches_scanned": 0,
        "event_files_found": 0,
        "missing_event_files": 0,
        "missing_event_paths": [],
        "event_file_read_errors": 0,
        "parsing_errors": 0,
        "error_examples": [],
        "open_play_passes_removed": 0,
        "shots_used": 0,
        "goals_used": 0,
        "successful_moves_used": 0,
    }


def load_events_for_match(match_id: int, counters: dict):
    path = event_file_path(match_id)
    if not path.exists():
        counters["missing_event_files"] += 1
        counters["missing_event_paths"].append(str(path))
        return None
    counters["event_files_found"] += 1
    try:
        return read_json_file(path)
    except Exception as exc:
        counters["event_file_read_errors"] += 1
        counters["parsing_errors"] += 1
        counters["error_examples"].append(f"{path}: {exc}")
        return None


def parse_relevant_event(event: dict, match_row: dict, counters: dict) -> dict | None:
    try:
        event_type = get_name(event.get("type"))
        if event_type not in {"Shot", "Pass", "Carry"}:
            return None
        location = event.get("location")
        if not valid_xy(location):
            return None

        team = event.get("team", {}) or {}
        player = event.get("player", {}) or {}
        action = {
            "match_id": match_row["match_id"],
            "competition_id": match_row["competition_id"],
            "competition_name": match_row["competition_name"],
            "country_name": match_row.get("country_name"),
            "season_id": match_row["season_id"],
            "season_name": match_row["season_name"],
            "team_id": team.get("id"),
            "team_name": team.get("name"),
            "player_id": player.get("id"),
            "player_name": player.get("name"),
            "event_type": event_type,
            "start_x": float(location[0]),
            "start_y": float(location[1]),
            "end_x": np.nan,
            "end_y": np.nan,
            "is_goal": False,
            "is_success": False,
            "move_type": None,
            "pass_type": None,
            "play_pattern": get_name(event.get("play_pattern")),
            "period": event.get("period"),
            "minute": event.get("minute"),
            "second": event.get("second"),
        }

        if event_type == "Shot":
            shot = event.get("shot", {}) or {}
            action["is_goal"] = get_name(shot.get("outcome")) == "Goal"
            return action

        if event_type == "Pass":
            if not INCLUDE_PASSES:
                return None
            pass_data = event.get("pass", {}) or {}
            pass_type = get_name(pass_data.get("type"))
            action["pass_type"] = pass_type
            if USE_OPEN_PLAY_ONLY and pass_type in SET_PIECE_PASS_TYPES:
                counters["open_play_passes_removed"] += 1
                return None
            end_location = pass_data.get("end_location")
            if not valid_xy(end_location):
                return None
            action["end_x"] = float(end_location[0])
            action["end_y"] = float(end_location[1])
            action["is_success"] = pass_data.get("outcome") is None
            action["move_type"] = "pass"
            return action

        if event_type == "Carry":
            if not INCLUDE_CARRIES:
                return None
            carry_data = event.get("carry", {}) or {}
            end_location = carry_data.get("end_location")
            if not valid_xy(end_location):
                return None
            action["end_x"] = float(end_location[0])
            action["end_y"] = float(end_location[1])
            action["is_success"] = True
            action["move_type"] = "carry"
            return action
    except Exception as exc:
        counters["parsing_errors"] += 1
        if len(counters["error_examples"]) < 10:
            counters["error_examples"].append(f"match {match_row.get('match_id')}: {exc}")
    return None

## Coordinate and direction quality check

All team actions need to be represented in the same attacking direction for xT matrices to be meaningful. StatsBomb open data is usually oriented toward the high-x attacking goal, but the shot-location check below makes that assumption visible. If shots are not mostly near high x values, add coordinate flipping before using the clustering output.

In [ ]:
def collect_shot_start_x_sample(metadata: pd.DataFrame, max_matches: int | None = 500) -> np.ndarray:
    counters = empty_processing_counters()
    shot_x_values = []
    rows = list(iter_match_rows(metadata))
    if max_matches is not None:
        rows = rows[:max_matches]
    for match_row in tqdm(rows, desc="Sampling shot locations"):
        counters["matches_scanned"] += 1
        events = load_events_for_match(match_row["match_id"], counters)
        if events is None:
            continue
        for event in events:
            action = parse_relevant_event(event, match_row, counters)
            if action is not None and action["event_type"] == "Shot":
                shot_x_values.append(action["start_x"])
    return np.array(shot_x_values, dtype=float)


shot_start_x_sample = collect_shot_start_x_sample(match_metadata)
if len(shot_start_x_sample) == 0:
    warnings.warn("No shots found in the coordinate sample. Direction quality could not be checked.")
else:
    quantiles = pd.Series(shot_start_x_sample).quantile([0.05, 0.25, 0.50, 0.75, 0.95])
    print("Shot start_x quantiles:")
    display(quantiles.to_frame("start_x"))
    if float(np.median(shot_start_x_sample)) < PITCH_LENGTH * 0.55:
        warnings.warn(
            "Shot locations are not mostly near high x values. Coordinates may not be normalized to a shared "
            "attacking left-to-right direction. TODO: add coordinate flipping before clustering."
        )
    else:
        print("Direction check looks reasonable: shots are mostly closer to the high-x attacking goal.")

## Train one global xT model

This first pass streams through event files and accumulates global shot counts, goal counts, successful movement counts, and transition counts. It then solves the xT recurrence with one shared transition matrix and saves `outputs/global_xt_zone_values.csv` and `outputs/global_xt_grid_12x16.csv`.

In [ ]:
def train_global_xt(metadata: pd.DataFrame) -> tuple[np.ndarray, pd.DataFrame, dict]:
    counters = empty_processing_counters()
    shot_counts = np.zeros(TOTAL_ZONES, dtype=float)
    goal_counts = np.zeros(TOTAL_ZONES, dtype=float)
    successful_move_counts = np.zeros(TOTAL_ZONES, dtype=float)
    transition_counts = np.zeros((TOTAL_ZONES, TOTAL_ZONES), dtype=float)

    for match_row in tqdm(list(iter_match_rows(metadata)), desc="First pass: global xT counts"):
        counters["matches_scanned"] += 1
        events = load_events_for_match(match_row["match_id"], counters)
        if events is None:
            continue
        for event in events:
            action = parse_relevant_event(event, match_row, counters)
            if action is None:
                continue
            start_zone = point_to_zone(action["start_x"], action["start_y"])
            if start_zone is None:
                continue
            if action["event_type"] == "Shot":
                shot_counts[start_zone] += 1
                counters["shots_used"] += 1
                if action["is_goal"]:
                    goal_counts[start_zone] += 1
                    counters["goals_used"] += 1
            elif action["move_type"] in {"pass", "carry"} and action["is_success"]:
                end_zone = point_to_zone(action["end_x"], action["end_y"])
                if end_zone is None:
                    continue
                successful_move_counts[start_zone] += 1
                transition_counts[start_zone, end_zone] += 1
                counters["successful_moves_used"] += 1

    action_counts = shot_counts + successful_move_counts
    shot_prob = np.divide(shot_counts, action_counts, out=np.zeros_like(shot_counts), where=action_counts > 0)
    move_prob = np.divide(successful_move_counts, action_counts, out=np.zeros_like(successful_move_counts), where=action_counts > 0)
    goal_prob = np.divide(goal_counts, shot_counts, out=np.zeros_like(goal_counts), where=shot_counts > 0)
    transition_matrix = np.divide(
        transition_counts,
        successful_move_counts[:, None],
        out=np.zeros_like(transition_counts),
        where=successful_move_counts[:, None] > 0,
    )

    xt = np.zeros(TOTAL_ZONES, dtype=float)
    immediate_shot_value = shot_prob * goal_prob
    max_delta = math.inf
    iterations_completed = 0
    for iteration in range(1, XT_MAX_ITERATIONS + 1):
        xt_next = immediate_shot_value + move_prob * transition_matrix.dot(xt)
        max_delta = float(np.max(np.abs(xt_next - xt)))
        xt = xt_next
        iterations_completed = iteration
        if max_delta < XT_CONVERGENCE_THRESHOLD:
            break

    xt_rows = []
    for zone in range(TOTAL_ZONES):
        x_bin, y_bin = zone_to_xy_bins(zone)
        xt_rows.append(
            {
                "zone": zone, "x_bin": x_bin, "y_bin": y_bin,
                "shot_count": shot_counts[zone],
                "goal_count": goal_counts[zone],
                "successful_move_count": successful_move_counts[zone],
                "shot_prob": shot_prob[zone],
                "move_prob": move_prob[zone],
                "goal_prob": goal_prob[zone],
                "xT": xt[zone],
            }
        )
    xt_df = pd.DataFrame(xt_rows)
    zone_values_path = OUTPUT_DIR / "global_xt_zone_values.csv"
    grid_path = OUTPUT_DIR / "global_xt_grid_12x16.csv"
    xt_df.to_csv(zone_values_path, index=False)
    pd.DataFrame(xt.reshape(GRID_W, GRID_L)).to_csv(grid_path, index=False, header=False)

    counters["iterations_completed"] = iterations_completed
    counters["final_max_delta"] = max_delta
    counters["global_xt_min"] = float(np.min(xt))
    counters["global_xt_max"] = float(np.max(xt))
    counters["global_xt_zone_values_path"] = str(zone_values_path)
    counters["global_xt_grid_path"] = str(grid_path)

    print(f"xT min: {np.min(xt):.8f}")
    print(f"xT max: {np.max(xt):.8f}")
    print(f"Iterations until stop: {iterations_completed}")
    print(f"Final max delta: {max_delta:.8g}")
    print(f"Shots used: {counters['shots_used']:,}")
    print(f"Goals used: {counters['goals_used']:,}")
    print(f"Successful moves used: {counters['successful_moves_used']:,}")
    print(f"Open-play pass removals: {counters['open_play_passes_removed']:,}")
    if np.any(xt < -1e-12):
        warnings.warn("Some global xT values are negative. Inspect the transition matrix and data filters.")
    return xt, xt_df, counters


global_xt, global_xt_table, training_counters = train_global_xt(match_metadata)
display(global_xt_table.sort_values("xT", ascending=False).head(10))

## Build team-season xT matrices

This second pass applies the global xT vector to every successful movement action. For each move, `xT_value = global_xt[end_zone] - global_xt[start_zone]` and `positive_xT = max(xT_value, 0)`.

In [ ]:
def build_base_team_season_table(metadata: pd.DataFrame) -> pd.DataFrame:
    home = metadata[["competition_id", "competition_name", "country_name", "season_id", "season_name", "match_id", "home_team_id", "home_team_name"]].rename(columns={"home_team_id": "team_id", "home_team_name": "team_name"})
    away = metadata[["competition_id", "competition_name", "country_name", "season_id", "season_name", "match_id", "away_team_id", "away_team_name"]].rename(columns={"away_team_id": "team_id", "away_team_name": "team_name"})
    team_matches = pd.concat([home, away], ignore_index=True).dropna(subset=["team_id"])
    team_matches["team_id"] = team_matches["team_id"].astype(int)
    return (
        team_matches.groupby(["competition_id", "competition_name", "country_name", "season_id", "season_name", "team_id", "team_name"], dropna=False)["match_id"]
        .nunique()
        .reset_index(name="match_count")
    )


def team_season_key(action: dict) -> tuple:
    return tuple(action[col] for col in TEAM_SEASON_KEYS)


def aggregate_team_season_matrices(metadata: pd.DataFrame, xt: np.ndarray):
    counters = empty_processing_counters()
    aggregates = {
        "created_start_positive": defaultdict(float),
        "received_end_positive": defaultdict(float),
        "created_start_net": defaultdict(float),
        "action_count_start": defaultdict(float),
        "pass_start_positive": defaultdict(float),
        "carry_start_positive": defaultdict(float),
    }
    team_totals = defaultdict(lambda: {col: 0.0 for col in TOTAL_COLUMNS})
    action_rows = [] if SAVE_ACTION_LEVEL_OUTPUT else None

    for match_row in tqdm(list(iter_match_rows(metadata)), desc="Second pass: team-season matrices"):
        counters["matches_scanned"] += 1
        events = load_events_for_match(match_row["match_id"], counters)
        if events is None:
            continue
        for event in events:
            action = parse_relevant_event(event, match_row, counters)
            if action is None or action["move_type"] not in {"pass", "carry"} or not action["is_success"]:
                continue
            start_zone = point_to_zone(action["start_x"], action["start_y"])
            end_zone = point_to_zone(action["end_x"], action["end_y"])
            if start_zone is None or end_zone is None:
                continue

            xt_value = float(xt[end_zone] - xt[start_zone])
            positive_xt = max(xt_value, 0.0)
            key = team_season_key(action)
            aggregates["created_start_positive"][(key, start_zone)] += positive_xt
            aggregates["received_end_positive"][(key, end_zone)] += positive_xt
            aggregates["created_start_net"][(key, start_zone)] += xt_value
            aggregates["action_count_start"][(key, start_zone)] += 1
            if action["move_type"] == "pass":
                aggregates["pass_start_positive"][(key, start_zone)] += positive_xt
            else:
                aggregates["carry_start_positive"][(key, start_zone)] += positive_xt

            totals = team_totals[key]
            totals["move_action_count"] += 1
            totals["total_net_xT"] += xt_value
            totals["total_positive_xT"] += positive_xt
            if action["move_type"] == "pass":
                totals["pass_count"] += 1
                totals["total_positive_pass_xT"] += positive_xt
            else:
                totals["carry_count"] += 1
                totals["total_positive_carry_xT"] += positive_xt
            counters["successful_moves_used"] += 1

            if action_rows is not None:
                action_rows.append({**{col: action[col] for col in TEAM_SEASON_KEYS}, "match_id": action["match_id"], "move_type": action["move_type"], "start_zone": start_zone, "end_zone": end_zone, "xT_start": xt[start_zone], "xT_end": xt[end_zone], "xT_value": xt_value, "positive_xT": positive_xt})

    base = build_base_team_season_table(metadata)
    totals_df = pd.DataFrame([dict(zip(TEAM_SEASON_KEYS, key)) | values for key, values in team_totals.items()])
    if totals_df.empty:
        totals_df = pd.DataFrame(columns=TEAM_SEASON_KEYS + TOTAL_COLUMNS)
    team_season = base.merge(totals_df, on=TEAM_SEASON_KEYS, how="left")
    for col in TOTAL_COLUMNS:
        team_season[col] = team_season[col].fillna(0)
    team_season["is_zero_positive_xT"] = team_season["total_positive_xT"] <= 0
    team_season["below_min_matches"] = team_season["match_count"] < MIN_MATCHES_PER_TEAM_SEASON
    team_season["below_min_positive_xT"] = team_season["total_positive_xT"] < MIN_POSITIVE_XT_PER_TEAM_SEASON

    action_level_df = pd.DataFrame(action_rows) if action_rows is not None else None
    if action_level_df is not None:
        action_path = OUTPUT_DIR / "team_season_action_level_xt_values.csv"
        action_level_df.to_csv(action_path, index=False)
        counters["action_level_output_path"] = str(action_path)
    return aggregates, team_season, counters, action_level_df


aggregates, team_season_metadata, matrix_counters, action_level_output = aggregate_team_season_matrices(match_metadata, global_xt)
print(f"Team-season rows before matrix filtering: {len(team_season_metadata):,}")
print(f"Successful movement actions aggregated: {matrix_counters['successful_moves_used']:,}")
display(team_season_metadata.sort_values("total_positive_xT", ascending=False).head(10))

## Matrix formatting and saved datasets

Each matrix is saved in long and/or wide format. Wide files contain one team-season row with `z000` through `z191`; long files contain one row per team-season-zone with `zone`, `x_bin`, `y_bin`, and `value`.

In [ ]:
def aggregate_dict_to_long(aggregate: dict, value_name: str = "value") -> pd.DataFrame:
    rows = []
    for (key, zone), value in aggregate.items():
        x_bin, y_bin = zone_to_xy_bins(zone)
        rows.append(dict(zip(TEAM_SEASON_KEYS, key)) | {"zone": zone, "x_bin": x_bin, "y_bin": y_bin, value_name: value})
    return pd.DataFrame(rows, columns=TEAM_SEASON_KEYS + ["zone", "x_bin", "y_bin", value_name])


def long_to_wide(long_df: pd.DataFrame, value_col: str = "value", prefix: str = "z") -> pd.DataFrame:
    if long_df.empty:
        wide = team_season_metadata[TEAM_SEASON_KEYS].copy()
        for col in zone_columns(prefix):
            wide[col] = 0.0
        return wide
    wide = long_df.pivot_table(index=TEAM_SEASON_KEYS, columns="zone", values=value_col, aggfunc="sum", fill_value=0.0).reset_index()
    for zone in range(TOTAL_ZONES):
        if zone not in wide.columns:
            wide[zone] = 0.0
    wide = wide[TEAM_SEASON_KEYS + list(range(TOTAL_ZONES))]
    return wide.rename(columns={zone: f"{prefix}{zone:03d}" for zone in range(TOTAL_ZONES)})


def normalize_wide_sum(wide_df: pd.DataFrame, prefix: str = "z") -> pd.DataFrame:
    feature_cols = zone_columns(prefix)
    normalized = wide_df.copy()
    totals = normalized[feature_cols].sum(axis=1)
    normalized[feature_cols] = normalized[feature_cols].div(totals.where(totals > 0), axis=0).fillna(0.0)
    normalized["distribution_sum"] = normalized[feature_cols].sum(axis=1)
    normalized["is_zero_distribution"] = totals <= 0
    return normalized


def distribution_long_from_wide(distribution_wide: pd.DataFrame, prefix: str = "z") -> pd.DataFrame:
    feature_cols = zone_columns(prefix)
    long = distribution_wide[TEAM_SEASON_KEYS + feature_cols].melt(id_vars=TEAM_SEASON_KEYS, value_vars=feature_cols, var_name="zone_feature", value_name="value")
    long["zone"] = long["zone_feature"].str.extract(r"(\d+)").astype(int)
    long["x_bin"] = long["zone"].mod(GRID_L)
    long["y_bin"] = long["zone"] // GRID_L
    return long[TEAM_SEASON_KEYS + ["zone", "x_bin", "y_bin", "value"]]


def save_matrix_outputs(long_df, wide_df, long_path, wide_path):
    if long_df is not None and long_path is not None:
        long_df.to_csv(OUTPUT_DIR / long_path, index=False)
    wide_df.to_csv(OUTPUT_DIR / wide_path, index=False)
    print(f"Saved {wide_path}: {wide_df.shape[0]:,} rows x {wide_df.shape[1]:,} columns")


created_raw_long = aggregate_dict_to_long(aggregates["created_start_positive"])
created_raw_wide = long_to_wide(created_raw_long)
created_distribution_wide = normalize_wide_sum(created_raw_wide)
created_distribution_long = distribution_long_from_wide(created_distribution_wide)

received_raw_long = aggregate_dict_to_long(aggregates["received_end_positive"])
received_raw_wide = long_to_wide(received_raw_long)
received_distribution_wide = normalize_wide_sum(received_raw_wide)

net_raw_wide = long_to_wide(aggregate_dict_to_long(aggregates["created_start_net"]))
action_count_wide = long_to_wide(aggregate_dict_to_long(aggregates["action_count_start"]))
pass_distribution_wide = normalize_wide_sum(long_to_wide(aggregate_dict_to_long(aggregates["pass_start_positive"])))
carry_distribution_wide = normalize_wide_sum(long_to_wide(aggregate_dict_to_long(aggregates["carry_start_positive"])))

save_matrix_outputs(created_raw_long, created_raw_wide, "team_season_xt_created_start_positive_raw_long.csv", "team_season_xt_created_start_positive_raw_wide.csv")
save_matrix_outputs(created_distribution_long, created_distribution_wide, "team_season_xt_created_start_positive_distribution_long.csv", "team_season_xt_created_start_positive_distribution_wide.csv")
save_matrix_outputs(None, received_raw_wide, None, "team_season_xt_received_end_positive_raw_wide.csv")
save_matrix_outputs(None, received_distribution_wide, None, "team_season_xt_received_end_positive_distribution_wide.csv")
save_matrix_outputs(None, net_raw_wide, None, "team_season_xt_created_start_net_raw_wide.csv")
save_matrix_outputs(None, action_count_wide, None, "team_season_action_count_start_wide.csv")
save_matrix_outputs(None, pass_distribution_wide, None, "team_season_xt_created_start_pass_positive_distribution_wide.csv")
save_matrix_outputs(None, carry_distribution_wide, None, "team_season_xt_created_start_carry_positive_distribution_wide.csv")

## Normalization quality checks

For clustering, sum-normalized positive xT created by start zone is the primary matrix. If a team-season has zero positive xT, its 192 features remain zero and the row is flagged.

In [ ]:
created_feature_cols = zone_columns("z")
nonzero_distribution = created_distribution_wide.loc[~created_distribution_wide["is_zero_distribution"], "distribution_sum"]
max_deviation = float((nonzero_distribution - 1).abs().max()) if len(nonzero_distribution) else 0.0
print(f"Max absolute deviation from distribution sum 1: {max_deviation:.10f}")

zero_rows = created_distribution_wide.loc[created_distribution_wide["is_zero_distribution"], TEAM_SEASON_KEYS + ["distribution_sum"]]
print(f"Rows with zero positive xT distribution: {len(zero_rows):,}")
if len(zero_rows):
    display(zero_rows.head(20))

low_match_rows = team_season_metadata.loc[team_season_metadata["below_min_matches"], TEAM_SEASON_METADATA_COLUMNS]
print(f"Rows with fewer than {MIN_MATCHES_PER_TEAM_SEASON} matches: {len(low_match_rows):,}")
if len(low_match_rows):
    display(low_match_rows.head(20))

low_positive_rows = team_season_metadata.loc[team_season_metadata["below_min_positive_xT"], TEAM_SEASON_METADATA_COLUMNS]
print(f"Rows below MIN_POSITIVE_XT_PER_TEAM_SEASON={MIN_POSITIVE_XT_PER_TEAM_SEASON}: {len(low_positive_rows):,}")
if len(low_positive_rows):
    display(low_positive_rows.head(20))

## Spatial summary features

The summary table describes where each team-season creates positive xT: defensive, middle, and attacking thirds; left, center, and right lanes; attacking-third lanes; and the top three zones.

In [ ]:
def spatial_summary_from_distribution(distribution_wide: pd.DataFrame) -> pd.DataFrame:
    feature_cols = zone_columns("z")
    zones = pd.DataFrame({"zone": np.arange(TOTAL_ZONES)})
    zones["x_bin"] = zones["zone"].mod(GRID_L)
    zones["y_bin"] = zones["zone"] // GRID_L

    def cols(mask):
        return [f"z{z:03d}" for z in zones.loc[mask, "zone"]]

    defensive = cols(zones["x_bin"].between(0, 4))
    middle = cols(zones["x_bin"].between(5, 10))
    attacking = cols(zones["x_bin"].between(11, 15))
    left = cols(zones["y_bin"].between(0, 3))
    center = cols(zones["y_bin"].between(4, 7))
    right = cols(zones["y_bin"].between(8, 11))
    left_attacking = sorted(set(left).intersection(attacking))
    center_attacking = sorted(set(center).intersection(attacking))
    right_attacking = sorted(set(right).intersection(attacking))

    rows = []
    for row in distribution_wide.itertuples(index=False):
        d = row._asdict()
        values = np.array([d[c] for c in feature_cols], dtype=float)
        order = np.argsort(values)[::-1]
        record = {col: d[col] for col in TEAM_SEASON_KEYS}
        record |= {
            "defensive_third_share": float(sum(d[c] for c in defensive)),
            "middle_third_share": float(sum(d[c] for c in middle)),
            "attacking_third_share": float(sum(d[c] for c in attacking)),
            "left_side_share": float(sum(d[c] for c in left)),
            "center_share": float(sum(d[c] for c in center)),
            "right_side_share": float(sum(d[c] for c in right)),
            "left_attacking_third_share": float(sum(d[c] for c in left_attacking)),
            "center_attacking_third_share": float(sum(d[c] for c in center_attacking)),
            "right_attacking_third_share": float(sum(d[c] for c in right_attacking)),
        }
        for rank in range(3):
            zone = int(order[rank])
            record[f"top_zone_{rank + 1}"] = zone
            record[f"top_zone_{rank + 1}_share"] = float(values[zone])
        rows.append(record)
    return pd.DataFrame(rows)


spatial_summary = spatial_summary_from_distribution(created_distribution_wide)
spatial_summary = spatial_summary.merge(team_season_metadata[TEAM_SEASON_METADATA_COLUMNS], on=TEAM_SEASON_KEYS, how="left")
spatial_summary_path = OUTPUT_DIR / "team_season_xt_spatial_summary.csv"
spatial_summary.to_csv(spatial_summary_path, index=False)
print(f"Saved {spatial_summary_path}: {spatial_summary.shape[0]:,} rows x {spatial_summary.shape[1]:,} columns")
display(spatial_summary.sort_values(["attacking_third_share", "total_positive_xT"], ascending=False).head(10))

## Combined clustering feature dataset

The final file keeps metadata first, then 192 created start-zone distribution features prefixed with `created_z`, 192 optional received end-zone distribution features prefixed with `received_z`, and compact summary features.

In [ ]:
def rename_zone_features(wide_df: pd.DataFrame, old_prefix: str, new_prefix: str) -> pd.DataFrame:
    return wide_df.rename(columns={f"{old_prefix}{z:03d}": f"{new_prefix}{z:03d}" for z in range(TOTAL_ZONES)})


created_features = rename_zone_features(created_distribution_wide[TEAM_SEASON_KEYS + zone_columns("z")], "z", "created_z")
received_features = rename_zone_features(received_distribution_wide[TEAM_SEASON_KEYS + zone_columns("z")], "z", "received_z")
summary_feature_cols = [
    "defensive_third_share", "middle_third_share", "attacking_third_share",
    "left_side_share", "center_share", "right_side_share",
    "left_attacking_third_share", "center_attacking_third_share", "right_attacking_third_share",
]

clustering_features = (
    team_season_metadata[TEAM_SEASON_METADATA_COLUMNS]
    .merge(created_features, on=TEAM_SEASON_KEYS, how="left")
    .merge(received_features, on=TEAM_SEASON_KEYS, how="left")
    .merge(spatial_summary[TEAM_SEASON_KEYS + summary_feature_cols], on=TEAM_SEASON_KEYS, how="left")
)
clustering_features["pass_xT_share"] = np.divide(
    clustering_features["total_positive_pass_xT"],
    clustering_features["total_positive_xT"],
    out=np.zeros(len(clustering_features), dtype=float),
    where=clustering_features["total_positive_xT"].to_numpy() > 0,
)
clustering_features["carry_xT_share"] = np.divide(
    clustering_features["total_positive_carry_xT"],
    clustering_features["total_positive_xT"],
    out=np.zeros(len(clustering_features), dtype=float),
    where=clustering_features["total_positive_xT"].to_numpy() > 0,
)
all_zone_feature_cols = zone_columns("created_z") + zone_columns("received_z")
clustering_features[all_zone_feature_cols] = clustering_features[all_zone_feature_cols].fillna(0.0)

clustering_path = OUTPUT_DIR / "team_season_xt_clustering_features.csv"
clustering_features.to_csv(clustering_path, index=False)
print(f"Saved {clustering_path}: {clustering_features.shape[0]:,} rows x {clustering_features.shape[1]:,} columns")

## Visual validation

This small validation section plots the global xT heatmap, 8 random eligible team-season normalized created surfaces, and the top 8 team-seasons by total positive xT.

In [ ]:
def maybe_smooth_grid(grid: np.ndarray) -> np.ndarray:
    if not SMOOTHING_OPTIONAL:
        return grid
    try:
        from scipy.ndimage import gaussian_filter
        return gaussian_filter(grid, sigma=0.8)
    except Exception as exc:
        warnings.warn(f"Smoothing requested but scipy was unavailable: {exc}")
        return grid


def vector_to_grid(values) -> np.ndarray:
    return np.asarray(values, dtype=float).reshape(GRID_W, GRID_L)


def plot_global_xt_heatmap(xt: np.ndarray, output_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(10, 6))
    image = ax.imshow(vector_to_grid(xt), origin="lower", cmap="viridis", aspect="auto")
    ax.set_title("Global xT zone values")
    ax.set_xlabel("x bin")
    ax.set_ylabel("y bin")
    fig.colorbar(image, ax=ax, label="xT")
    fig.tight_layout()
    fig.savefig(output_path, dpi=160)
    plt.show()


def plot_team_surfaces(surface_wide: pd.DataFrame, selected_rows: pd.DataFrame, title: str, output_path: Path) -> None:
    feature_cols = zone_columns("z")
    if selected_rows.empty:
        warnings.warn(f"No rows available for {title}")
        return
    selected = selected_rows.head(8).merge(surface_wide, on=TEAM_SEASON_KEYS, how="left")
    n = len(selected)
    ncols = 4
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.4 * nrows), squeeze=False)
    vmax = max(float(selected[feature_cols].to_numpy().max()), 1e-12)
    for ax, row in zip(axes.flat, selected.itertuples(index=False)):
        d = row._asdict()
        grid = maybe_smooth_grid(vector_to_grid([d[col] for col in feature_cols]))
        image = ax.imshow(grid, origin="lower", cmap="magma", aspect="auto", vmin=0, vmax=vmax)
        ax.set_title(f"{d['team_name']}\n{d['competition_name']} {d['season_name']}", fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
    for ax in axes.flat[n:]:
        ax.axis("off")
    fig.suptitle(title, fontsize=14)
    fig.colorbar(image, ax=axes.ravel().tolist(), shrink=0.75, label="Share of positive xT")
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.show()


global_heatmap_path = OUTPUT_DIR / "global_xt_heatmap.png"
sample_surfaces_path = OUTPUT_DIR / "sample_team_season_xt_surfaces.png"
top_surfaces_path = OUTPUT_DIR / "top_team_season_xt_surfaces.png"

plot_global_xt_heatmap(global_xt, global_heatmap_path)
eligible_rows = team_season_metadata.loc[
    (~team_season_metadata["below_min_matches"])
    & (~team_season_metadata["below_min_positive_xT"])
    & (~team_season_metadata["is_zero_positive_xT"])
].copy()
sample_rows = eligible_rows.sample(n=min(8, len(eligible_rows)), random_state=RANDOM_SEED) if len(eligible_rows) else eligible_rows
plot_team_surfaces(created_distribution_wide, sample_rows, "Random sample of normalized team-season created xT surfaces", sample_surfaces_path)
top_rows = team_season_metadata.sort_values("total_positive_xT", ascending=False).head(8)
plot_team_surfaces(created_distribution_wide, top_rows, "Top team-seasons by total positive xT", top_surfaces_path)

print(f"Saved {global_heatmap_path}")
print(f"Saved {sample_surfaces_path}")
print(f"Saved {top_surfaces_path}")

## Data leakage and clustering note

The global xT model creates a shared value scale across all team-seasons. The sum-normalized team-season matrix is better for style clustering than raw total xT because it reduces the effect of team strength, possession volume, and number of matches. Raw xT matrices are still useful for ranking attacking production, but they are not ideal as direct clustering input unless the goal is to cluster by strength and production.

Use distribution features in `outputs/team_season_xt_clustering_features.csv` for tactical-style clustering.

## Processing log and final output summary

The final cell writes `outputs/xt_matrix_build_log.txt` and prints the shape of the main clustering dataset.

In [ ]:
def write_processing_log() -> Path:
    end_time = datetime.now()
    output_files = sorted(str(path) for path in OUTPUT_DIR.glob("team_season_*.csv"))
    output_files += sorted(str(path) for path in OUTPUT_DIR.glob("global_xt*.csv"))
    output_files += sorted(str(path) for path in OUTPUT_DIR.glob("*xt*.png"))
    lines = [
        "Team-season xT matrix build log",
        f"start_time: {training_counters['start_time']}",
        f"end_time: {end_time}",
        f"matches_scanned_training_pass: {training_counters['matches_scanned']}",
        f"matches_scanned_matrix_pass: {matrix_counters['matches_scanned']}",
        f"event_files_found_training_pass: {training_counters['event_files_found']}",
        f"event_files_found_matrix_pass: {matrix_counters['event_files_found']}",
        f"missing_event_files_training_pass: {training_counters['missing_event_files']}",
        f"missing_event_files_matrix_pass: {matrix_counters['missing_event_files']}",
        f"parsing_errors_training_pass: {training_counters['parsing_errors']}",
        f"parsing_errors_matrix_pass: {matrix_counters['parsing_errors']}",
        f"open_play_passes_removed_training_pass: {training_counters['open_play_passes_removed']}",
        f"open_play_passes_removed_matrix_pass: {matrix_counters['open_play_passes_removed']}",
        f"shots_used: {training_counters['shots_used']}",
        f"goals_used: {training_counters['goals_used']}",
        f"successful_moves_used_training_pass: {training_counters['successful_moves_used']}",
        f"successful_moves_used_matrix_pass: {matrix_counters['successful_moves_used']}",
        f"team_season_rows_produced: {len(team_season_metadata)}",
        "",
        "Output files:",
    ]
    lines.extend(f"- {path}" for path in output_files)
    if training_counters["error_examples"] or matrix_counters["error_examples"]:
        lines.extend(["", "Error examples:"])
        lines.extend(f"- {err}" for err in training_counters["error_examples"][:10])
        lines.extend(f"- {err}" for err in matrix_counters["error_examples"][:10])
    log_path = OUTPUT_DIR / "xt_matrix_build_log.txt"
    log_path.write_text("\n".join(lines), encoding="utf-8")
    return log_path


log_path = write_processing_log()
metadata_columns = TEAM_SEASON_METADATA_COLUMNS
spatial_feature_count = len(zone_columns("created_z"))
print("Final notebook output summary")
print(f"Team-season rows: {len(clustering_features):,}")
print(f"Spatial features: {spatial_feature_count:,} created start-zone distribution features")
print(f"Metadata columns: {len(metadata_columns):,}")
print(f"Main clustering dataset: {clustering_path}")
print(f"Processing log: {log_path}")

print("\nGenerated core files:")
for path in [
    OUTPUT_DIR / "match_metadata.csv",
    OUTPUT_DIR / "global_xt_zone_values.csv",
    OUTPUT_DIR / "global_xt_grid_12x16.csv",
    OUTPUT_DIR / "team_season_xt_created_start_positive_raw_long.csv",
    OUTPUT_DIR / "team_season_xt_created_start_positive_raw_wide.csv",
    OUTPUT_DIR / "team_season_xt_created_start_positive_distribution_long.csv",
    OUTPUT_DIR / "team_season_xt_created_start_positive_distribution_wide.csv",
    OUTPUT_DIR / "team_season_xt_received_end_positive_raw_wide.csv",
    OUTPUT_DIR / "team_season_xt_received_end_positive_distribution_wide.csv",
    OUTPUT_DIR / "team_season_xt_created_start_net_raw_wide.csv",
    OUTPUT_DIR / "team_season_action_count_start_wide.csv",
    OUTPUT_DIR / "team_season_xt_created_start_pass_positive_distribution_wide.csv",
    OUTPUT_DIR / "team_season_xt_created_start_carry_positive_distribution_wide.csv",
    OUTPUT_DIR / "team_season_xt_spatial_summary.csv",
    OUTPUT_DIR / "team_season_xt_clustering_features.csv",
    OUTPUT_DIR / "global_xt_heatmap.png",
    OUTPUT_DIR / "sample_team_season_xt_surfaces.png",
    OUTPUT_DIR / "top_team_season_xt_surfaces.png",
    OUTPUT_DIR / "xt_matrix_build_log.txt",
]:
    print(f"- {path}")